# Predict heat-illness from the heat

**Day 3 — a regression lab.** Titanic was a **yes/no** question (survived or not) — that is *classification*. This time we predict an actual **number** — that is *regression*.

**What we predict**: how many people are taken to hospital by ambulance for **heat illness** on a given day, in a given prefecture of Japan.

**Where the data comes from** (nothing is invented):
- Transports (the target) = **Japan's Fire and Disaster Management Agency (FDMA / 総務省消防庁)**, official daily × prefecture records.
- Temperature & humidity = **Open-Meteo historical archive** (ERA5 reanalysis).
- 2018–2024 summers (May–Sep), 7 climate-diverse prefectures (Hokkaido, Tokyo, Aichi, Osaka, **Hyogo = Kobe, where this course is held**, Fukuoka, Okinawa).

## Step 1 — Load the data

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/itoksk/summer-ai-materials/main/materials/data/heatstroke.csv"
df = pd.read_csv(url)
print(df.shape)
df.head()

## Step 2 — Look first

Before any model, look at the link between heat and transports with your own eyes.

- `tmax_c`: daily max temperature | `transported`: people transported (the target) | `pref_en`: prefecture | `severe`: severe + fatal cases

In [ ]:
# mean transports per 3°C temperature bin
df["tmax_bin"] = (df["tmax_c"] // 3 * 3).astype(int)
print(df.groupby("tmax_bin")["transported"].mean().round(1))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.scatter(df["tmax_c"], df["transported"], s=6, alpha=0.15)
plt.xlabel("daily max temperature (°C)")
plt.ylabel("people transported (per prefecture/day)")
plt.title("Heat vs heat-illness transports")
plt.show()

**Observe**: Straight line, or does it **bend upward** past ~30°C? That non-linear jump is the key to today.

## Step 3 — Predict BEFORE you train

Write your own guess first:

1. On a day when the max temperature hits **35°C**, how many people do you think one prefecture sends to hospital? Write a number.
2. Besides temperature, what else might drive the count (prefecture / month / humidity ...)? Name one.

No peeking — the model reveals the real answer in a moment.

## Step 4 — Train on the past, predict an unseen summer

To prevent cheating, split **by year**: train on 2018–2023, then predict **2024 (a summer the model has never seen)**. That is what "predict on the test set" means.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

train = df[df["year"] < 2024]
test  = df[df["year"] == 2024]
print("train rows:", len(train), " / test rows (2024):", len(test))

# simplest possible model: a straight line from tmax only
lin = LinearRegression().fit(train[["tmax_c"]], train["transported"])
pred = lin.predict(test[["tmax_c"]])
print("linear MAE:", round(mean_absolute_error(test["transported"], pred), 2), "people")
at35 = pd.DataFrame({"tmax_c": [35]})
print("prediction at 35°C:", round(float(lin.predict(at35)[0]), 1), "people  <- compare with your Step 3 guess")

**Question**: Did the straight line capture the surge on the hottest days (35°C+)? It probably **under-predicts** them, because the real relationship bends but a line cannot.

## Step 5 — A better model with more clues

A **random forest** (many decision trees voting) handles the bend. Give it not only temperature but also **prefecture, month, humidity**.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

feat = ["tmax_c", "tmean_c", "humidity_pct", "month"]
Xtr = pd.get_dummies(train[feat + ["pref_en"]], columns=["pref_en"])
Xte = pd.get_dummies(test[feat + ["pref_en"]], columns=["pref_en"]).reindex(columns=Xtr.columns, fill_value=0)

rf = RandomForestRegressor(n_estimators=200, random_state=42).fit(Xtr, train["transported"])
pred_rf = rf.predict(Xte)
print("RandomForest MAE:", round(mean_absolute_error(test["transported"], pred_rf), 2), "people")
print("(smaller than the linear MAE? = it predicts better)")

## Step 6 — Answer-check on a real day

Take the **hottest Tokyo day of 2024**, let the model predict it, and compare with what **actually** happened.

In [ ]:
test = test.copy()
test["pred"] = pred_rf.round(0)

tokyo = test[test["pref_en"] == "Tokyo"].sort_values("tmax_c", ascending=False)
print(tokyo[["date", "tmax_c", "transported", "pred"]].head(5).to_string(index=False))
print("\nHow close was the prediction to reality? By how much was it off?")

In [ ]:
# which clues mattered most?
imp = sorted(zip(Xtr.columns, rf.feature_importances_), key=lambda t: -t[1])
print("feature importance (top 8):")
for name, v in imp[:8]:
    print(f"  {name:18} {round(v, 3)}")
print("\nDid the factor you guessed in Step 3 show up?")

## Step 7 — The threshold and the alert

Transports explode above ~30°C. That is why Japan issues a **Heat-Illness Alert** (環境省 / JMA) when the **WBGT heat index** crosses a set line. The "elbow" the model found is essentially the same line the country uses to protect people.

## Reflection & limits

- The model uses each prefecture **capital city's** temperature as a stand-in for the whole (large) prefecture. Same warning as Titanic: **a model only knows what you measured.**
- Bigger-population prefectures have more transports (Tokyo ≫ Okinawa). To compare fairly, you would normalise **per 100,000 residents**.
- Summers keep getting hotter (climate change). On a future hotter than anything in the training data, will the model's predictions still hold?
- **Challenge**: (1) normalise per capita to compare prefectures fairly (2) add your own prefecture (FDMA has all 47) (3) reshape it into a **classification** task — "should an alert be issued tomorrow, yes/no?"

**Sources**
- FDMA "Heatstroke emergency transport status" https://www.fdma.go.jp/disaster/heatstroke/
- Open-Meteo Historical Weather API (ERA5) https://open-meteo.com/
- Ministry of the Environment heat-index (WBGT) site https://www.wbgt.env.go.jp/